In [1]:
import utils
import visualization
import numpy as np
import os
from IPython.display import display, Video
import ctp_swap
from manavlib.common.params import (
    ExperimentParams,
    BaseDiscreteAgentParams,
    BaseAlgParams,
)
import manavlib.io.xml_io as new_io
from manavlib.gen.maps import reduce_cellsize

In [2]:
%load_ext line_profiler

In [3]:
TASK_SUFFIX = "_task.xml"  # Set task file naming convention with a suffix (_task.xml).
TASK_DIR = "../tasks/room-64-64-16"  # Define the directory path where task files are located.
MAP_PATH = os.path.join(
    TASK_DIR, "map.xml"
)  # Specify the paths for the map and configuration files.
CONFIG_PATH = os.path.join(TASK_DIR, f"ctp_swap_config.xml")
TASK_ID = 1  # Select TASK_ID for the experiment, indicating the specific task to load.

In [4]:
agents_num = 10  # Set the number of agents for the experiment.
exp_params, alg_params = new_io.read_xml_config(
    CONFIG_PATH
)  # Read experiment and algorithm parameters from the XML configuration file.
h, w, cs, grid_map, obstacles = new_io.read_xml_map(
    MAP_PATH
)  # Load map data from the XML map file.

exp_params.max_steps = 2000  # Set the maximum number of steps for the experiment.

In [5]:
task_file = f"{TASK_ID}{TASK_SUFFIX}"
task_path = os.path.join(TASK_DIR, task_file)

# Read the start positions, goal positions, and agent-specific parameters from the XML task file.
default_params, starts, goals, ag_params = new_io.read_xml_agents(task_path)
starts = starts[:, 0:2]
starts = starts[:agents_num, 0:2]
goals = goals[:, 0:2]
goals = goals[:agents_num, 0:2]

for ap in ag_params:
    ap.size = 0.25

In [6]:
ag_params_obj = ctp_swap.convert_agent_params(ag_params[0])

(NavAlg, nav_params), (Planner, planner_params), (Follower, follower_params) = (
    utils.get_algorithms(alg_params)
)

planner_params_obj = ctp_swap.convert_alg_params(planner_params)
vis_graph_obj = ctp_swap.VisibilityGraph(obstacles, goals, 0.7)

In [7]:
planner  = ctp_swap.VisibilityPlanner(ag_params_obj, planner_params_obj, goals, vis_graph_obj)

In [8]:
print(utils.Summary.header())

simulation = utils.Simulation(
    starts,
    goals[:agents_num],
    grid_map,
    cs,
    obstacles,
    planner,
    agents_num,
    ag_params,
    alg_params,
    exp_params,
    True,
)


summary, steps_log, goal_log, neighbors_log = simulation.run_experiment()

print(summary)

success  collision  collision_obst   makespan   flowtime    runtime     mean_groups     mean_groups_size     number
      1          0               0    102.100    437.600      0.276           9.553                 1.05         10


In [9]:
output_dir = "img"
output_filename = "animated_trajectories"
output_ext = "mp4"
output_path = os.path.join(output_dir, f"{output_filename}.{output_ext}")


grid_map_obj = ctp_swap.GridMap(grid_map, cs)
grid_map_obj.inflate(ag_params[0].size)
grid_map_inf = grid_map_obj.get_map()

visualization.draw(
    grid_map,
    None,
    cs,
    obstacles,
    goals,
    steps_log,
    ag_params,
    goal_log,
    neighbors_log,
    25,
    output_path,
)

In [10]:
display(Video(filename=output_path))